# Phase 6a — final training, resumable across sessions

Trains the frozen recipe on both declared variants at three seeds: **six runs**.

**This notebook does not unblind anything.** Training does not spend the one-shot
external evaluation — that is Phase 6b, a separate notebook, run once after the
pre-registration is committed.

## Built to be interrupted

Six runs at roughly 3.6 GPU-h each exceed a single week's quota, so the notebook
assumes it will be stopped and restarted:

- Runs execute in a **fixed order, grouped by variant**, so a session that ends early
  leaves a whole variant finished at all three seeds.
- Each run is **priced before it starts** and skipped if it will not fit the remaining
  budget. Stopping cleanly beats being killed mid-epoch — that epoch is lost and the
  quota does not refund it.
- Every run checkpoints each epoch and resumes with `--resume`.

## The step that is easy to skip and expensive to skip

`/kaggle/working` **does not survive between sessions.** At the end of every session,
publish the output as **`verify-dr-phase6`**; at the start of the next, attach it.
Section 6 carries those checkpoints forward. Miss it and completed runs are retrained
from zero, silently.

## Inputs

`verify-dr-cache-512`, `verify-dr-manifests`, and from the second session on
`verify-dr-phase6`.

| Setting | Value |
|---|---|
| Accelerator | **GPU T4 x2** |
| Persistence | **Files only** |
| Internet | **On** |
| Environment | Pin to original environment |

## 1 · Clone the repo

In [ ]:
import shutil, sys
from pathlib import Path

REPO_DIR = Path("/kaggle/working/repo")
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)          # always take a clean checkout

# Private repo? Store a GitHub PAT under Add-ons -> Secrets as GH_TOKEN.
# Public repo? Delete the try/except and just clone the plain URL.
url = "https://github.com/kazimab1/DR-New.git"
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GH_TOKEN")
    url = url.replace("https://", f"https://{token}@")
    print("cloning with GH_TOKEN")
except Exception:
    print("no GH_TOKEN secret found - cloning anonymously (works if the repo is public)")

!git clone --depth 1 -b claude/charming-faraday-a02cx9 {url} {REPO_DIR} 2>&1 | tail -2

sys.path.insert(0, str(REPO_DIR / "scripts"))
sys.path.insert(0, str(REPO_DIR / "src"))
assert (REPO_DIR / "scripts/build_cache.py").exists(), "clone failed - check the token or branch name"

# Drop verify_dr modules left over from an earlier checkout in this kernel.
# Python caches modules by NAME, not by file, so re-cloning mid-session does
# nothing for an already-imported package: a later cell importing a function
# added upstream still fails with ImportError against the new files on disk.
for _stale in [m for m in list(sys.modules)
               if m == "verify_dr" or m.startswith("verify_dr.")]:
    del sys.modules[_stale]

# Print the commit actually in use. A stale checkout is the single most common
# cause of a confusing failure downstream: the notebook cell is new, the scripts
# on disk are not.
import subprocess
_sha = subprocess.run(["git", "-C", str(REPO_DIR), "log", "-1", "--format=%h  %s"],
                      capture_output=True, text=True).stdout.strip()
print("repo ready at", REPO_DIR)
print("checked out:", _sha)

## 2 · GPU check

In [ ]:
import torch
print('torch', torch.__version__, '| cuda', torch.version.cuda)
if not torch.cuda.is_available():
    raise RuntimeError("No GPU. Set Accelerator to 'GPU T4 x2' and re-run from the top.")
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f'  [{i}] {p.name}  {p.total_memory / 2**30:.1f} GB')

## 3 · Helpers

Same helpers as Phase 4, including the streaming `run()` — a three-hour job that prints nothing until it exits cannot be stopped in epoch 1 when it is going wrong.

In [ ]:
from pathlib import Path
from collections import Counter
import json, shlex, subprocess, sys, time, zipfile

INPUT = Path("/kaggle/input")
WORK = Path("/kaggle/working")
RESULTS = WORK / "results"
RESULTS.mkdir(parents=True, exist_ok=True)

def q(x):
    return shlex.quote(str(x))

def run(cmd):
    """Run a training job, streaming its output live.

    The earlier notebooks use capture_output=True, which is fine for a two-minute
    manifest build and useless here: you would see nothing at all until a
    three-hour job exited. Streaming means a per-epoch line appears as it happens,
    so a run that is going wrong can be stopped in epoch 1 rather than hour 3.
    """
    print("$", cmd, flush=True)
    proc = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        print(line.rstrip(), flush=True)
    code = proc.wait()
    if code != 0:
        if code == 2:
            print("\nExit 2 is an argument error. Usually the cloned scripts are stale:")
            print("re-run the clone cell at the top, then run from there.")
        raise RuntimeError(f"training failed with exit {code}")

def cache_roots():
    """Every directory that looks like a build_cache.py output root.

    build_cache.py writes <root>/<dataset>/cache_report.json, so a report file
    identifies its root two levels up. /kaggle/input is searched first: a stale
    copy in /kaggle/working must never silently win over the dataset you attached.
    """
    found = []
    for base in (INPUT, WORK):
        if not base.exists():
            continue
        for root, _ in Counter(r.parent.parent for r in base.rglob("cache_report.json")).most_common():
            found.append(root)
    return found

def resolve_datasets(roots):
    """dataset -> the root holding the best copy of it.

    The cache is legitimately split across published datasets: the full build
    plus a later top-up. IDRiD's lesion masks ship as their own dataset
    ('verify-dr-idrid-masks'), so a single root shows idrid with zero mask
    channels even when the masks are attached. When a dataset appears in more
    than one root, the copy with more mask channels wins.
    """
    best = {}
    for root in roots:
        for d in sorted(x for x in root.iterdir() if x.is_dir()):
            if not (d / "cache_report.json").exists():
                continue
            masks = d / "masks"
            score = len(list(masks.iterdir())) if masks.is_dir() else 0
            if d.name not in best or score > best[d.name][1]:
                best[d.name] = (root, score)
    return {name: root for name, (root, _) in best.items()}

def extract_cache(dest=WORK / "cache512"):
    """Extract a cache published as a zip. Idempotent within a session.

    This costs GPU-session minutes, which come out of the 30 h/week quota. If you
    hit it every run, re-upload the cache to Kaggle as a *dataset* rather than as
    notebook output -- an uploaded zip is unpacked by Kaggle once, server-side.
    """
    for z in sorted(INPUT.rglob("*.zip")):
        try:
            with zipfile.ZipFile(z) as zf:
                names = zf.namelist()
        except (zipfile.BadZipFile, OSError):
            continue
        if not any(n.endswith("cache_report.json") for n in names):
            continue
        marker = dest / ".extracted_from"
        if marker.exists() and marker.read_text().strip() == z.name:
            print(f"already extracted from {z.name}")
            return dest
        print(f"extracting {z.name} ({z.stat().st_size / 2**30:.1f} GB) -> {dest}")
        dest.mkdir(parents=True, exist_ok=True)
        started = time.time()
        with zipfile.ZipFile(z) as zf:
            zf.extractall(dest)
        marker.write_text(z.name)
        print(f"extracted in {(time.time() - started) / 60:.1f} min")
        return dest
    return None

def find_manifest_dir():
    """Phase 2's output: the directory holding dataset_plan.json and the variants."""
    for base in (INPUT, WORK):
        if not base.exists():
            continue
        hits = sorted(base.rglob("dataset_plan.json"))
        if hits:
            return hits[0].parent
    return None

## 4 · Cache and manifests

In [ ]:
ROOTS = cache_roots()
if not ROOTS and extract_cache():
    ROOTS = cache_roots()
MANIFEST_DIR = find_manifest_dir()

if not ROOTS:
    raise RuntimeError(
        "No cache found. Attach verify-dr-cache-512 (and verify-dr-idrid-masks) "
        "under Add Data in the right-hand panel.")
if MANIFEST_DIR is None:
    raise RuntimeError(
        "No manifests found. Attach the Phase 2 output (verify-dr-manifests), or "
        "add 02_manifests.ipynb as a notebook input.")

DATASETS = resolve_datasets(ROOTS)
print("cache roots  :")
for r in ROOTS:
    print("   ", r)
print("manifest dir :", MANIFEST_DIR)

IMAGE_SUFFIXES = {".jpg", ".jpeg", ".png", ".tif", ".tiff", ".bmp"}

print("\ndatasets in the cache:")
print(f"  {'dataset':<12}{'images':>8}  {'masks':>5}  root")
for name, root in sorted(DATASETS.items()):
    d = root / name
    images = d / 'images'
    # rglob, not glob: glob('*') counts direct children only, so a nested
    # layout reports 1 and looks like catastrophic data loss when nothing
    # is actually wrong.
    n_img = sum(1 for f in images.rglob('*') if f.suffix.lower() in IMAGE_SUFFIXES) \
        if images.is_dir() else 0
    n_msk = len([m for m in (d / 'masks').iterdir() if m.is_dir()]) \
        if (d / 'masks').is_dir() else 0
    print(f"  {name:<12}{n_img:>8}  {n_msk:>5}  {root}")

    if images.is_dir():
        subdirs = [x for x in images.glob('*') if x.is_dir()]
        if subdirs and n_img:
            print(f"               ^ nested under {len(subdirs)} subdirectorie(s), "
                  f"e.g. {subdirs[0].name}/ - fine, the manifest stores full paths")
    if n_img == 0:
        print(f"               ^ NO IMAGES - {name} is empty in every attached root")

print("\nmanifests available:")
for c in sorted(MANIFEST_DIR.glob("*.csv")):
    print("  ", c.name)

# Phase 3 could ignore this column. Phase 4 cannot: C2 IS the mask experiment.
mask_bearing = [n for n, r in sorted(DATASETS.items())
                if (r / n / "masks").is_dir()
                and any(x.is_dir() for x in (r / n / "masks").iterdir())]
print()
if mask_bearing:
    print(f"mask channels present in: {mask_bearing}")
else:
    print("NO dataset in any attached root has mask channels.")
    print("C2 and C3 cannot run. Attach verify-dr-idrid-masks (or verify-dr-idrid)")
    print("alongside verify-dr-cache-512 -- IDRiD's masks ship separately from the")
    print("main cache. Section 5 below re-checks this per experiment.")

# Every root goes to --cache-root, so each dataset resolves to the root that
# actually holds it instead of all of them being forced onto one.
CACHE_FLAGS = ' '.join(q(r) for r in ROOTS)

## 5 · This session's budget

In [ ]:
# ---------------------------------------------------------------------------
# How much GPU Kaggle says you have left THIS WEEK (Settings -> Accelerator).
# Set it honestly. The guard below refuses to START a run it cannot finish, and
# an optimistic number buys nothing: a run killed mid-epoch loses that epoch and
# the quota does not refund it.
# ---------------------------------------------------------------------------
GPU_HOURS_LEFT = 11.9

THROUGHPUT_IMG_S = 46.0   # measured on B1's 512 px T4 runs, not guessed
SAFETY_MARGIN_H  = 0.5    # session startup, carry-forward copy, saving output

import time
# Kaggle bills WALL-CLOCK while the GPU is attached. Cache extraction, the
# carry-forward copy and any idle cell draw on the same quota -- not just the
# seconds spent inside a training call. The guard in section 8 therefore counts
# from here, not from the sum of run durations.
SESSION_START = time.time()

RESULTS.mkdir(parents=True, exist_ok=True)
print(f'budget: {GPU_HOURS_LEFT:.1f} h   usable after margin: '
      f'{GPU_HOURS_LEFT - SAFETY_MARGIN_H:.1f} h')
print('clock started: this session is billed from now, not from section 8.')

## 6 · Carry forward completed runs

**Skipping this from session 2 onward costs a full session of GPU.**

In [ ]:
# Bring forward whatever an earlier session already trained.
#
# /kaggle/working does NOT survive between sessions. Every completed run must come
# back in as an attached dataset or it is silently retrained from scratch --
# roughly 3.6 GPU-h thrown away per run.
import shutil

prior = None
for cand in sorted(INPUT.glob('*/**/results')) + sorted(INPUT.glob('*/results')):
    if any(cand.glob('H1_*/checkpoint.pt')) or any(cand.glob('H1_*/metrics.json')):
        prior = cand
        break

if prior is None:
    print('No previous Phase 6 output attached.')
    print('Correct for the first session. From the second on, attach verify-dr-phase6')
    print('under Add Data -- otherwise finished runs are trained again.')
else:
    print(f'carrying forward from {prior}')
    for run_dir in sorted(x for x in prior.iterdir() if x.is_dir()):
        dest = RESULTS / run_dir.name
        if dest.exists():
            print(f'   {run_dir.name:<26} already here, left alone')
            continue
        shutil.copytree(run_dir, dest)
        state = 'complete' if (dest / 'metrics.json').exists() else 'partial -> will resume'
        print(f'   {run_dir.name:<26} {state}')

## 7 · The six runs, and what is left

In [ ]:
import json as _json
import pandas as pd

# The six runs in a FIXED order. Order is what makes "we finished the first three"
# a statement about WHICH three, across sessions.
#
# Grouped by variant rather than interleaved by seed, so any session that ends
# early leaves a whole variant done at all three seeds -- an analysable
# intermediate state instead of a scatter.
VARIANTS = ['eyepacs_full', 'eyepacs_ddr_full']
SEEDS    = [42, 43, 44]
PLAN     = [(v, s) for v in VARIANTS for s in SEEDS]

# The frozen recipe, preregistration/frozen_config.yaml, stated IN FULL.
# Nothing is left to a script default: train_grading.py defaults to 12 epochs and
# Stage B selected this recipe at 10. A default that drifts turns six runs into
# six runs of something else, and nothing in the output would say so.
FROZEN = dict(image_size=512, backbone='efficientnet_b0', head='ordinal_focal',
              focal_gamma=2.0, sampler='stratified_exposure', epochs=10,
              batch_size=32, lr=3e-4, weight_decay=1e-4, dropout=0.3,
              warmup_epochs=2, patience=3)

rows, total_left, missing = [], 0.0, []
for variant, seed in PLAN:
    name = f'H1_{variant}_s{seed}'
    path = MANIFEST_DIR / f'{variant}.csv'

    if not path.exists():
        missing.append(path.name)
        rows.append({'run': name, 'train rows': 0, 'cost h': 0.0,
                     'state': 'MANIFEST MISSING', 'left h': 0.0})
        continue

    f = pd.read_csv(path)
    n_train = int((f['split'] == 'train').sum()) if 'split' in f.columns else len(f)
    hours = n_train * FROZEN['epochs'] / THROUGHPUT_IMG_S / 3600

    done = RESULTS / name / 'metrics.json'
    hist = RESULTS / name / 'history.json'
    if done.exists():
        state, left = 'done', 0.0
    elif (RESULTS / name / 'checkpoint.pt').exists():
        ep = len(_json.loads(hist.read_text())) if hist.exists() else 0
        state, left = f'partial ({ep}/{FROZEN["epochs"]} epochs)', \
                      hours * max(0.0, 1 - ep / FROZEN['epochs'])
    else:
        state, left = 'not started', hours

    total_left += left
    rows.append({'run': name, 'train rows': n_train, 'cost h': round(hours, 2),
                 'state': state, 'left h': round(left, 2)})

print(pd.DataFrame(rows).to_string(index=False))

if missing:
    raise RuntimeError(
        f'missing manifest(s): {missing}\n'
        'Phase 6 trains the two declared variants. Run scripts/build_variants.py '
        '(notebook 02) and attach its output as verify-dr-manifests.')

usable = GPU_HOURS_LEFT - SAFETY_MARGIN_H
print()
print(f'remaining work: {total_left:.1f} GPU-h      usable budget: {usable:.1f} h')
if total_left > usable:
    print(f'-> does not fit; about {-(-total_left // usable):.0f} more session(s).')
    print('   Expected. The loop below runs what fits and stops cleanly.')
else:
    print('-> everything remaining fits in this session.')
print()
print('These are UPPER bounds: early stopping (patience 3 on val QWK) can end a')
print('run before epoch 10, which only ever frees budget.')

## 8 · Run whatever fits

In [ ]:
budget = GPU_HOURS_LEFT - SAFETY_MARGIN_H


def spent_h():
    """Hours billed so far this session.

    Elapsed wall-clock, not the sum of training calls: everything above section 8
    was billed too. It also self-corrects -- if 46 img/s proves optimistic, the
    overrun shows up after the first run and the guard stops, instead of
    confidently starting two more that were never going to fit.
    """
    return (time.time() - SESSION_START) / 3600

completed, remaining = [], []
print(f'{spent_h():.2f} h billed before any training started.')

for idx, ((variant, seed), row) in enumerate(zip(PLAN, rows)):
    name, need = row['run'], row['left h']

    if row['state'] == 'done':
        print(f'{name}: complete already, skipping')
        completed.append(name)
        continue

    if spent_h() + need > budget:
        print(f'\n{name}: needs {need:.2f} h, only {budget - spent_h():.2f} h left -- STOP.')
        print('  Deliberately not started rather than started and killed: a run cut')
        print('  off mid-epoch loses that epoch and the quota does not refund it.')
        remaining = [r['run'] for r in rows[idx:] if r['state'] != 'done']
        break

    print('\n' + '=' * 72)
    print(f'{name}   ~{need:.2f} h   ({budget - spent_h():.2f} h of budget left)')
    print('=' * 72, flush=True)

    started = time.time()
    run(' '.join([
        f"python {q(REPO_DIR / 'scripts/train_grading.py')}",
        f"--manifest {q(MANIFEST_DIR / f'{variant}.csv')}",
        f'--experiment {q(name)}',
        f'--results-dir {q(RESULTS)}',
        f'--cache-root {CACHE_FLAGS}',
        f"--backbone {FROZEN['backbone']} --head {FROZEN['head']}",
        f"--sampler {FROZEN['sampler']} --focal-gamma {FROZEN['focal_gamma']}",
        f"--image-size {FROZEN['image_size']} --batch-size {FROZEN['batch_size']}",
        f"--epochs {FROZEN['epochs']} --warmup-epochs {FROZEN['warmup_epochs']}",
        f"--lr {FROZEN['lr']} --weight-decay {FROZEN['weight_decay']}",
        f"--dropout {FROZEN['dropout']} --patience {FROZEN['patience']}",
        f'--seed {seed} --workers 4 --resume',
    ]))

    took = (time.time() - started) / 3600
    completed.append(name)
    print(f'  {name}: {took:.2f} h actual vs {need:.2f} estimated')

print('\n' + '=' * 72)
print(f'completed this session : {completed or "none"}')
print(f'still to do            : {remaining or "none"}')
print(f'GPU billed this session: {spent_h():.2f} h of {budget:.2f} usable')
print('=' * 72)

## 9 · Where Phase 6 stands

In [ ]:
import json as _json
import pandas as pd

table = []
for variant, seed in PLAN:
    name = f'H1_{variant}_s{seed}'
    path = RESULTS / name / 'metrics.json'
    if not path.exists():
        table.append({'run': name, 'status': 'not run'})
        continue
    m = _json.loads(path.read_text())
    b = m['best_val']
    table.append({
        'run': name, 'status': 'done',
        'QWK': round(b['qwk'], 4),
        'macro-F1': round(b['macro_f1'], 4),
        # per_class_f1 is keyed by STRING grade, not indexed by position.
        'g1-F1': round(b['per_class_f1']['1'], 4),
        'g1-recall': round(b['per_class_recall']['1'], 4),
        'MAE': round(b['mae'], 4),
        'distinct': b['distinct_predictions'],
        'epochs': m['epochs_run'],
        'min': round(m['minutes']),
    })

print(pd.DataFrame(table).to_string(index=False))
done = [t for t in table if t['status'] == 'done']

# A collapsed run decides nothing, whatever its QWK looks like: a model that never
# predicts grade 1 still scores ~0.97 QWK, because grade 1 sits one step from 0.
collapsed = [t['run'] for t in done if t.get('distinct') == 1]
if collapsed:
    print(f'\nWARNING: {collapsed} predicted a single grade for every image.')
    print('Those runs decide nothing and must be re-run, not reported.')

print()
if len(done) < len(PLAN):
    print(f'{len(done)} of {len(PLAN)} runs complete -- publish and continue next session.')
else:
    print('All six runs complete. Phase 6 TRAINING is done.')
    print()
    print('This is NOT the unblinding. The single external evaluation -- EyePACS')
    print('official test, APTOS, Messidor-2 -- is a separate notebook, run ONCE,')
    print('after the pre-registration is committed and acknowledged. Fit temperature')
    print('on the internal calibration split first. There is no second attempt.')

---
## 10 · Save — every session, no exceptions

1. **Save Version → Save & Run All (Commit).**
2. Publish `/kaggle/working/results` as **`verify-dr-phase6`** — a new version each
   session.
3. **Attach that dataset next session** so section 6 can carry the checkpoints forward.
4. Record each completed run in `docs/04_experiment_register.md` under **H1**: QWK,
   macro-F1, grade-1 F1, `distinct_predictions`, per seed and variant.

> **Phase 6b, the unblinding, is deliberately not in this notebook.** It evaluates the
> locked sets **once**, and only after the pre-registration is committed and
> acknowledged. Keeping it separate means re-running this notebook can never trigger it.